# Holcomb Smoke Scrape


This notebook runs a Central Pinnacles subtree scrape using the package code. It is set up to validate whether we capture the expected 192 climbs and the route-level fields you care about, including path, grade, type, GPS, FA, page views, description, location, protection, and comments.


Recommended kernel: the workspace `.venv`.

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'src').exists() else CWD.parent
SRC = PROJECT_ROOT / 'src'
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_ROOT = PROJECT_ROOT / 'data'
EXPORT_ROOT = DATA_ROOT / 'exports'

PROJECT_ROOT

PosixPath('/Users/peterwilliams/projects/mountainproject/notebooks')

In [ ]:
from mountainproject.scraper.client import MountainProjectClient
from mountainproject.scraper.crawler import CrawlOptions, MountainProjectCrawler
from mountainproject.storage.catalog import ExportCatalog
from mountainproject.storage.exporters import JsonExporter
AREA_URLS = {
    'holcomb_root': 'https://www.mountainproject.com/area/105805238/holcomb-valley-pinnacles',
    'central_pinnacles': 'https://www.mountainproject.com/area/105931166/central-pinnacles',
    'pinnacles_east': 'https://www.mountainproject.com/area/105931181/pinnacles-east',
    'pinnacles_south': 'https://www.mountainproject.com/area/105931169/pinnacles-south',
    'pinnacles_north': 'https://www.mountainproject.com/area/105931172/pinnacles-north',
}

TARGET_NAME = 'central_pinnacles'
START_URL = AREA_URLS[TARGET_NAME]
OUTPUT_DIR = EXPORT_ROOT / TARGET_NAME

EXPECTED_ROUTE_COUNT = 192
REQUIRED_ROUTE_FIELDS = [
    'breadcrumbs',
    'yds_grade',
    'route_type_raw',
    'latitude',
    'longitude',
    'fa',
    'page_views_total',
    'description',
    'location',
    'protection',
    'comments',
]

DELAY_SECONDS = 0.0
MAX_DEPTH = 2
FETCH_COMMENTS = True
FETCH_ROUTE_STATS = False
RESOLVE_PHOTO_PAGES = False
DOWNLOAD_IMAGES = False
SAVE_HTML = False

print(START_URL)
print(OUTPUT_DIR)

https://www.mountainproject.com/area/105931166/central-pinnacles
/Users/peterwilliams/projects/mountainproject/notebooks/data/holcomb_smoke/central_pinnacles


## Scope and rate note

`MAX_DEPTH = 2` is deliberate here because Central Pinnacles is not a flat area page. Its climbs are distributed across child crags and a few grandchild subareas.

`DELAY_SECONDS` is set to `0.0` so you can test quickly. For a larger live crawl, raise it substantially and stay conservative.

`FETCH_COMMENTS = True` means this notebook will also hit the route comments endpoint. If you want a faster coverage-only pass first, set it to `False` and rerun.

In [ ]:
client = MountainProjectClient(
    delay_seconds=DELAY_SECONDS,
    cache_dir=OUTPUT_DIR / '.cache' / 'http',
    user_agent='Mozilla/5.0',
)
exporter = JsonExporter(
    OUTPUT_DIR,
    reuse_catalog=ExportCatalog(root=EXPORT_ROOT, current_output_dir=OUTPUT_DIR),
)
crawler = MountainProjectCrawler(
    client=client,
    exporter=exporter,
    options=CrawlOptions(
        max_depth=MAX_DEPTH,
        fetch_comments=FETCH_COMMENTS,
        fetch_route_stats=FETCH_ROUTE_STATS,
        skip_existing_route_stats=True,
        reuse_existing_data=True,
        resolve_photo_pages=RESOLVE_PHOTO_PAGES,
        download_images=DOWNLOAD_IMAGES,
        save_html=SAVE_HTML,
    ),
    log=print,
 )
manifest = crawler.crawl_area_tree(START_URL)
manifest

Scraping area: https://www.mountainproject.com/area/105931166/central-pinnacles
Scraping route: https://www.mountainproject.com/route/105878952/gold-standard
Scraping route: https://www.mountainproject.com/route/105805463/bye-crackie
Scraping route: https://www.mountainproject.com/route/105805473/coyotes-at-sunset
Scraping route: https://www.mountainproject.com/route/105874831/shoot-at-will
Scraping route: https://www.mountainproject.com/route/105805488/black-magic-poodle
Scraping route: https://www.mountainproject.com/route/105805492/golden-poodle
Scraping route: https://www.mountainproject.com/route/105813004/ricochet
Scraping route: https://www.mountainproject.com/route/105822594/claim-jumper
Scraping route: https://www.mountainproject.com/route/105822598/one-armed-bandit
Scraping route: https://www.mountainproject.com/route/105930868/powder-keg
Scraping route: https://www.mountainproject.com/route/105807952/pistol-pete
Scraping route: https://www.mountainproject.com/route/105807964

{'start_url': 'https://www.mountainproject.com/area/105931166/central-pinnacles',
 'started_at': '2026-06-22T23:31:20.454465+00:00',
 'finished_at': '2026-06-22T23:33:36.712078+00:00',
 'max_depth': 2,
 'fetch_comments': True,
 'resolve_photo_pages': False,
 'download_images': False,
 'save_html': False,
 'counts': {'areas': 23, 'routes': 207, 'comments': 377, 'photos': 824}}

In [10]:
import json

manifest_path = OUTPUT_DIR / 'manifest.json'
manifest_data = json.loads(manifest_path.read_text())
manifest_data

{'start_url': 'https://www.mountainproject.com/area/105931166/central-pinnacles',
 'started_at': '2026-06-22T23:31:20.454465+00:00',
 'finished_at': '2026-06-22T23:33:36.712078+00:00',
 'max_depth': 2,
 'fetch_comments': True,
 'resolve_photo_pages': False,
 'download_images': False,
 'save_html': False,
 'counts': {'areas': 23, 'routes': 207, 'comments': 377, 'photos': 824}}

In [11]:
import pandas as pd

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_json(path, lines=True)

areas = read_jsonl(OUTPUT_DIR / 'areas.jsonl')
routes = read_jsonl(OUTPUT_DIR / 'routes.jsonl')
comments = read_jsonl(OUTPUT_DIR / 'comments.jsonl')
photos = read_jsonl(OUTPUT_DIR / 'photos.jsonl')

if not routes.empty:
    routes['full_area_path'] = routes['breadcrumbs'].apply(
        lambda xs: ' > '.join(xs) if isinstance(xs, list) else None
    )
    routes['full_route_path'] = routes.apply(
        lambda row: ' > '.join(
            (row['breadcrumbs'] if isinstance(row['breadcrumbs'], list) else []) + [row['name']]
        ),
        axis=1,
    )
    routes['comment_rows_in_route_json'] = routes['comments'].apply(
        lambda xs: len(xs) if isinstance(xs, list) else 0
    )

{
    'area_rows': len(areas),
    'route_rows': len(routes),
    'comment_rows': len(comments),
    'photo_rows': len(photos),
}

{'area_rows': 23, 'route_rows': 207, 'comment_rows': 377, 'photo_rows': 824}

In [12]:
def missing_count(frame: pd.DataFrame, column: str) -> int:
    if frame.empty or column not in frame.columns:
        return 0
    return int(frame[column].isna().sum())

routes_outside_central = 0
if not routes.empty and 'breadcrumbs' in routes.columns:
    routes_outside_central = int(
        (~routes['breadcrumbs'].apply(lambda xs: isinstance(xs, list) and 'Central Pinnacles' in xs)).sum()
    )

validation = {
    'expected_route_count': EXPECTED_ROUTE_COUNT,
    'actual_route_count': len(routes),
    'route_count_matches': len(routes) == EXPECTED_ROUTE_COUNT,
    'routes_outside_central_pinnacles': routes_outside_central,
    'missing_breadcrumbs': missing_count(routes, 'breadcrumbs'),
    'missing_yds_grade': missing_count(routes, 'yds_grade'),
    'missing_route_type_raw': missing_count(routes, 'route_type_raw'),
    'missing_latitude': missing_count(routes, 'latitude'),
    'missing_longitude': missing_count(routes, 'longitude'),
    'missing_fa': missing_count(routes, 'fa'),
    'missing_page_views_total': missing_count(routes, 'page_views_total'),
    'missing_description': missing_count(routes, 'description'),
    'missing_location': missing_count(routes, 'location'),
    'missing_protection': missing_count(routes, 'protection'),
    'missing_comments_column': int('comments' not in routes.columns),
    'routes_with_truncated_comments': int(routes['comments_truncated'].fillna(False).sum()) if not routes.empty and 'comments_truncated' in routes.columns else 0,
    'route_json_comment_rows': int(routes['comment_rows_in_route_json'].sum()) if not routes.empty and 'comment_rows_in_route_json' in routes.columns else 0,
    'comments_jsonl_rows': len(comments),
}

validation

{'expected_route_count': 192,
 'actual_route_count': 207,
 'route_count_matches': False,
 'routes_outside_central_pinnacles': 15,
 'missing_breadcrumbs': 0,
 'missing_yds_grade': 0,
 'missing_route_type_raw': 0,
 'missing_latitude': 0,
 'missing_longitude': 0,
 'missing_fa': 0,
 'missing_page_views_total': 0,
 'missing_description': 0,
 'missing_location': 32,
 'missing_protection': 0,
 'missing_comments_column': 0,
 'routes_with_truncated_comments': 61,
 'route_json_comment_rows': 361,
 'comments_jsonl_rows': 377}

In [14]:
routes[[
    'route_id',
    'name',
    'full_route_path',
    'yds_grade',
    'route_type_raw',
    'latitude',
    'longitude',
    'fa',
    'page_views_total',
    'description',
    'location',
    'protection',
    'comment_count',
    'comments_truncated',
]]#.head(25) if not routes.empty else routes

,route_id,name,full_route_path,yds_grade,route_type_raw,latitude,longitude,fa,page_views_total,description,location,protection,comment_count,comments_truncated
0,105878952,Gold Standard,All Locations > California > San Bernardino Mo...,5.6 YDS,"Sport, Alpine",34.307701,-116.878346,"Chris Miller & Lisa Guindon, March 2002",18754,Incut plates and edges past two bolts lead to ...,Located on the far right side of the Gold Wall...,"7 bolts, chain anchors\n(all bolts 1/2"" SS)\nB...",11,True
1,105805463,Bye Crackie,All Locations > California > San Bernardino Mo...,5.7 YDS,"Sport, Alpine",34.307686,-116.878077,"Jim Hammerle and Rick Shull, 1989",17459,This high-quality route was one of the very fi...,This is the right-most bolted climb on the cra...,"6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",14,True
2,105805473,Coyotes at Sunset,All Locations > California > San Bernardino Mo...,5.8 YDS,"Sport, Alpine",34.303754,-116.878481,"(TR) Bob Cable & Julia Cronk, 1988, FL: Kevin ...",15456,"This route, like it's neighbor to the right, i...",This is the second bolted route from the left ...,"6 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",13,True
3,105874831,Shoot at Will,All Locations > California > San Bernardino Mo...,5.8 YDS,"Sport, Alpine",34.307902,-116.878203,"Jim Hammerle, Rick Shull & Dave Masuo, 1989; F...",7916,Starts in a recessed area and wanders up the f...,Located on an East-facing wall around and outs...,"5 bolts, chain anchor",8,True
4,105805488,Black Magic Poodle,All Locations > California > San Bernardino Mo...,5.9 YDS,"Sport, Alpine",34.307798,-116.878104,"Chris Miller and Loren Scott, June 1997",7206,Start by stemming until possible to grab holds...,This is the left-most climb on the crag and ju...,"8 bolts, bolted anchor/rap",9,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,114534895,Miner's Milk,All Locations > California > San Bernardino Mo...,5.9 YDS,"Trad, Alpine",34.308688,-116.878149,"Reed Ames & Chris Miller, June 2018",1096,Jam and lieback a curving flake system then co...,Located on the far right side of the wall just...,"Gear to 2.5 inches, 1 bolt, bolted sport ancho...",0,False
203,105931043,Motherlode,All Locations > California > San Bernardino Mo...,5.11b YDS,"Sport, Alpine",34.308688,-116.878149,"Chris Miller & Jake Colella, May 2000",2765,A boulder problem start (crux) on thin edges w...,Up a black streak between\nGolden Nugget\nand\...,"5 bolts, chain anchor\n(all bolts 1/2"" SS)\n* ...",3,False
204,114534865,Phantom Ore Cart,All Locations > California > San Bernardino Mo...,5.10b YDS,"Sport, Alpine",34.308688,-116.878149,"Chris Miller and Reed Ames, June 2018",1043,Tricky thin face moves off the ground and past...,Located on the far right side of the wall betw...,"4 bolts, bolted sport anchor (shared with\nMin...",0,False
205,117840035,The Prospector,All Locations > California > San Bernardino Mo...,5.8 YDS,"Sport, Alpine",34.308688,-116.878149,"(TR) Unknown; First Lead: Bryn Owen, October 2019",1257,"Start in a shallow left facing dihedral, then ...","Just to the right of Wildrose, before going ar...",5 bolts to a shared anchor with Wildrose.\n(bo...,2,False
